# Download the dataset

Week 1 task: download the Genpact food demand dataset and keep a backup.

Dataset: https://www.kaggle.com/datasets/kannanaikkal/food-demand-forecasting

You need a Kaggle API key in `~/.kaggle/kaggle.json` first (see README).

In [1]:
import os
import hashlib
import pandas as pd

In [2]:
DATA_DIR = '../data/raw'
os.makedirs(DATA_DIR, exist_ok=True)

# check the kaggle key is there
key_path = os.path.expanduser('~/.kaggle/kaggle.json')
print('kaggle key found:', os.path.exists(key_path))

kaggle key found: True


## Download

In [3]:
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()

# skip if already downloaded
if os.path.exists(DATA_DIR + '/train.csv'):
    print('already downloaded')
else:
    api.dataset_download_files('kannanaikkal/food-demand-forecasting', path=DATA_DIR, unzip=True)
    print('done')

already downloaded


In [4]:
for f in sorted(os.listdir(DATA_DIR)):
    if f.endswith('.csv'):
        size = os.path.getsize(os.path.join(DATA_DIR, f)) / 1e6
        print(f, round(size, 2), 'MB')

fulfilment_center_info.csv 0.0 MB
meal_info.csv 0.0 MB
sample_submission.csv 0.33 MB
test.csv 1.27 MB
train.csv 18.73 MB


## Checksums

Saving md5 of each file so I can check later that the raw files were not changed.

In [5]:
lines = []
for f in sorted(os.listdir(DATA_DIR)):
    if f.endswith('.csv'):
        with open(os.path.join(DATA_DIR, f), 'rb') as fh:
            h = hashlib.md5(fh.read()).hexdigest()
        lines.append(h + '  ' + f)
        print(h, f)

with open(DATA_DIR + '/CHECKSUMS.md5', 'w') as out:
    out.write('\n'.join(lines) + '\n')

36b68b4909d55d286babaf29181ff5c7 fulfilment_center_info.csv
0e6d793e7f04e633b5b801d5565d118f meal_info.csv
f6bcabae2dde4ac6ce9b40afff8e88b3 sample_submission.csv
39d846a6f03df93403b96f5ef7ece3fa test.csv


606b3838d46c5c64837bed648267c319 train.csv


## Quick check

Compare with the numbers in the proposal / Panda and Mohanty (2023): about 456k rows, 145 weeks, 77 centres,
51 meals (the paper says 50).

In [6]:
train = pd.read_csv(DATA_DIR + '/train.csv')
centers = pd.read_csv(DATA_DIR + '/fulfilment_center_info.csv')
meals = pd.read_csv(DATA_DIR + '/meal_info.csv')

print('train:', train.shape)
print('centers:', centers.shape)
print('meals:', meals.shape)
train.head()

train: (456548, 9)
centers: (77, 5)
meals: (51, 3)


,id,week,center_id,meal_id,checkout_price,base_price,emailer_for_promotion,homepage_featured,num_orders
0,1379560,1,55,1885,136.83,152.29,0,0,177
1,1466964,1,55,1993,136.83,135.83,0,0,270
2,1346989,1,55,2539,134.86,135.86,0,0,189
3,1338232,1,55,2139,339.50,437.53,0,0,54
4,1448490,1,55,2631,243.50,242.50,0,0,40


In [7]:
print('weeks:', train['week'].min(), 'to', train['week'].max())
print('centers in train:', train['center_id'].nunique())
print('meals in train:', train['meal_id'].nunique())
print('centre-meal series:', train.groupby(['center_id', 'meal_id']).ngroups)

weeks: 1 to 145
centers in train: 77
meals in train: 51
centre-meal series: 3597


In [8]:
train.isnull().sum()

id                       0
week                     0
center_id                0
meal_id                  0
checkout_price           0
base_price               0
emailer_for_promotion    0
homepage_featured        0
num_orders               0
dtype: int64

In [9]:
train['num_orders'].describe()

count    456548.000000
mean        261.872760
std         395.922798
min          13.000000
25%          54.000000
50%         136.000000
75%         324.000000
max       24299.000000
Name: num_orders, dtype: float64

In [10]:
# test.csv is from the hackathon, no target so I can't use it
test = pd.read_csv(DATA_DIR + '/test.csv')
print(test.shape, 'weeks', test['week'].min(), '-', test['week'].max())
print('num_orders in test?', 'num_orders' in test.columns)

(32573, 8) weeks 146 - 155
num_orders in test? False


## Notes

- numbers match: 456,548 rows, weeks 1-145, 77 centres, 51 meals -> use 51 in the report, not 50
- no missing values
- 3,597 series x 145 weeks = 521,565, but only 456,548 rows, so about 65k centre-meal-weeks are missing.
  Looked at this in notebook 02.
- smallest num_orders is 13, no zeros
- num_orders is very skewed (mean 262, median 136, max 24,299)